# Usage examples of the time-dependent dual TRILEX package

## N-point contour Green's functions

In [4]:

import sys
sys.path.append('..')
import numpy as np
from itertools import product

from triqs.gf import MeshReTime  # TRIQS real time mesh
from triqs.gf import MeshProduct # A direct product of 1D meshes

from tddt.keldysh import KeldyshGF # Green's function container on a 2-branch Keldysh contour

### Construct some (zero) Keldysh Green's function objects

In [5]:
t_max = 5.0
N_t = 51

t_mesh = MeshReTime(0.0, t_max, N_t)  # A 1D real time grid

2-point contour functions

In [6]:
tt_mesh = MeshProduct(t_mesh, t_mesh) # A 2D mesh as a direct product of t_mesh with itself

# A scalar-valued function 
g_scalar = KeldyshGF(mesh=tt_mesh)

# A matrix-valued function
# Each of the two time arguments has a single discrete index (e.g. spin projection) associated with it.
# The index runs over two values 0, 1.
g_matrix = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2,), (2,)))

# A tensor-valued function
# Now, we have 2 indices associated with each time argument (e.g. spin projection and orbital index)
# Those two indices vary in the ranges 0, 1 and 0, 1, 2 correspondingly
g_tensor = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 3), (2, 3)))

2-point contour functions with an extra k-argument

In [7]:
# Import some TRIQS modules related to lattice
from triqs.lattice import BravaisLattice, BrillouinZone
from triqs.gf import MeshBrillouinZone

lat = BravaisLattice(units=[(1, 0, 0), (0, 1, 0)])  # 2D square lattice
bz = BrillouinZone(lat)  # Brillouin zone of the lattice

n_k = 10 # Number of k-points along each dimension
bz_mesh = MeshBrillouinZone(bz, n_k) # k-mesh on 1BZ

# A product of two real-time grids and a k-mesh
ttk_mesh = MeshProduct(t_mesh, t_mesh, bz_mesh)

g_k_scalar = KeldyshGF(mesh=ttk_mesh)
g_k_matrix = KeldyshGF(mesh=ttk_mesh, arg_index_shapes=((2,), (2,)))
g_k_tensor = KeldyshGF(mesh=ttk_mesh, arg_index_shapes=((2, 3), (2, 3)))

3-point contour functions (vertices)

In [8]:
ttt_mesh = MeshProduct(t_mesh, t_mesh, t_mesh)

# Scalar-valued 
v_scalar = KeldyshGF(mesh=ttt_mesh)

# One extra discrete index per time argument
v_1 = KeldyshGF(mesh=ttt_mesh, arg_index_shapes=((2,), (2,), (2,)))

# Two extra discrete indices per time argument
v_2 = KeldyshGF(mesh=ttt_mesh, arg_index_shapes=((2, 3), (2, 3), (2, 3)))

# The first two time arguments have 2 discrete indices attached to each of them.
# The last time argument has only one discrete index (e.g. the bosonic channel)
v_3 = KeldyshGF(mesh=ttt_mesh, arg_index_shapes=((2, 3), (2, 3), (4,)))

1-point contour functions, e.g. time-dependent dispersion

In [9]:
f_scalar = KeldyshGF(mesh=t_mesh)

# f_i(t), i=0,1
f_i = KeldyshGF(mesh=t_mesh, arg_index_shapes=((2,),))

# f_{ij}(t), i=0,1, j =0, ..., 4
f_ij = KeldyshGF(mesh=t_mesh, arg_index_shapes=((2, 5),))

### Access real-time components of `KeldyshGF`

In [10]:
from tddt.keldysh import Branch

FW = Branch.FORWARD
BW = Branch.BACKWARD

# 2-point GF
print(g_matrix[FW, FW])  # Keldysh component g^{++}(t, t') as a TRIQS Gf object defined on MeshProduct(t_mesh, t_mesh)
print(g_matrix[FW, BW])  # Keldysh component g^{+-}(t, t')

# Vertex
print(v_3[FW, BW, BW])   # Keldysh component v^{+--}(t, t', t'')

Greens Function  with mesh Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5 and target_shape (2, 2): 

Greens Function  with mesh Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5 and target_shape (2, 2): 

Greens Function  with mesh Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5 and target_shape (2, 3, 2, 3, 4): 



Extract $G^<(t,t')$, $G^>(t,t')$, $G^{ret}(t,t')$ and $G^{adv}(t,t')$

In [11]:
from tddt.keldysh import lesser, greater, retarded, advanced

print(lesser(g_tensor))
print(greater(g_tensor))
print(retarded(g_tensor))
print(advanced(g_tensor))

Greens Function  with mesh Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5 and target_shape (2, 3, 2, 3): 

Greens Function  with mesh Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5 and target_shape (2, 3, 2, 3): 

Greens Function  with mesh Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5 and target_shape (2, 3, 2, 3): 

Greens Function  with mesh Real Time Mesh of size 51, t_min: 0, t_max: 5, Real Time Mesh of size 51, t_min: 0, t_max: 5 and target_shape (2, 3, 2, 3): 



Access a value corresponding to a single mesh point.

In [30]:
from tddt.keldysh import ContourPoint

# Points on the real time axis 
t_points = list(t_mesh)
#print(t_points)
t1, t2 = t_points[0], t_points[3]

# Contour points
z1 = ContourPoint(Branch.FORWARD, t1)
z2 = ContourPoint(Branch.BACKWARD, t2)

print(z1.t,z2.t,z1.branch,z2.branch)

print(g_scalar[z1, z2])
print(g_matrix[z1, z2])

# Set a single value in a matrix-valued GF
g_matrix[z1, z2] = np.eye(2)

print(g_matrix[z1, z2][0, 0])
print(g_matrix[z1, z2][0, 1])

mesh_point(linear_index = 0, value = 0.0) mesh_point(linear_index = 3, value = 0.30000000000000004) Branch.FORWARD Branch.BACKWARD
0j
[[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]
(1+0j)
0j


### Factory functions for `KeldyshGF`

Construct a 2-point contour function from its lesser and greater components.

In [13]:
from triqs.gf import Gf
from tddt.keldysh import from_lesser_greater

from scipy.linalg import expm  # Matrix exponential

H = np.array([[1.0, 0.5j], [-0.5j, 2.0]])
n = 0.1

g_l = Gf(mesh=tt_mesh, target_shape=(2, 2))
g_g = Gf(mesh=tt_mesh, target_shape=(2, 2))

for t1, t2 in tt_mesh:
    g_g[t1, t2] = -1j * (1.0 - n) * expm(-1j * H * (t1 - t2))
    g_l[t1, t2] = -1j * (-n) * expm(-1j * H * (t1 - t2))

g = from_lesser_greater(g_l, g_g)

Construct a 3-point vertex from 6 real time correlators.

Each element of dictionary G corresponds to one permutation of operators
in the correlator,
$$
    G_{ijk}(t_0, t_1, t_2) = -\xi_{ijk} \langle O_i(t_i) O_j(t_j) O_k(t_k)\rangle,
$$

where $O_0(t_0) = c(t_0)$, $O_1(t_1) = c^\dagger(t_1)$, $O_2(t_2) = \rho(t_2)$.
$\xi_{ijk} = -1$ if permutation (ijk) swaps indices 0 and 1, and +1 otherwise.

Keys are 3! = 6 triplets (i, j, k), which are permutations of (0, 1, 2) indicating the respective order of $c$, $c^\dagger$ and $\rho$.

In [14]:
from tddt.keldysh import from_vertex3_pieces

def make_time_piece(x):
    g = Gf(mesh=ttt_mesh, target_shape=())
    g.data[:] = x
    return g

G = {(0, 1, 2): make_time_piece(1.0),  # G_{012}(t_0, t_1, t_2)
     (0, 2, 1): make_time_piece(2.0),  # G_{021}(t_0, t_1, t_2)
     (1, 0, 2): make_time_piece(3.0),  # G_{102}(t_0, t_1, t_2)
     (1, 2, 0): make_time_piece(4.0),  # G_{120}(t_0, t_1, t_2)
     (2, 0, 1): make_time_piece(5.0),  # G_{201}(t_0, t_1, t_2)
     (2, 1, 0): make_time_piece(6.0)}  # G_{210}(t_0, t_1, t_2)

Lambda = from_vertex3_pieces(G)

### Hermitian conjugate of a 2-point contour function.

$C^\ddagger_{a,b}(z, z')$ is a Hermitian conjugate of $C_{a,b}(z, z')$ if
$$
\begin{array}{ll}
    [C^\ddagger]^<_{a,b}(t, t') &= -[C^<_{b,a}(t', t)]^*, \\
    [C^\ddagger]^>_{a,b}(t, t') &= -[C^>_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{ret}_{a,b}(t, t') &= [C^{adv}_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{adv}_{a,b}(t, t') &= [C^{ret}_{b,a}(t', t)]^*
\end{array}
$$
$a$ and $b$ are multi-indices associated with $t$ and $t'$.

In [15]:
from tddt.keldysh import herm_conj, is_hermitian

def make_keldysh_gf(H):
    g_l = Gf(mesh=tt_mesh, target_shape=(2, 2))
    g_g = Gf(mesh=tt_mesh, target_shape=(2, 2))
    
    n = 0.1
    for t1, t2 in tt_mesh:
        g_g[t1, t2] = -1j * (1.0 - n) * expm(-1j * H * (t1 - t2))
        g_l[t1, t2] = -1j * (-n) * expm(-1j * H * (t1 - t2))
        
    return from_lesser_greater(g_l, g_g)

#
# Prepare a non-Hermitian 2-point contour function
#

g = make_keldysh_gf(np.array([[1.0, 0.5j], [0, 2.0]]))
g_herm = make_keldysh_gf(np.array([[1.0, 0.5j], [-0.5j, 2.0]]))

print("g is Hermitian:", is_hermitian(g))
print("g_herm is Hermitian:", is_hermitian(g_herm))

g_l, g_g = lesser(g), greater(g)
g_ret, g_adv = retarded(g), advanced(g)

g_hc = herm_conj(g)

g_hc_l, g_hc_g = lesser(g_hc), greater(g_hc)
g_hc_ret, g_hc_adv = retarded(g_hc), advanced(g_hc)
        
print(all((g_hc_l[t1, t2] == -g_l[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_g[t1, t2] == -g_g[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_ret[t1, t2] == g_adv[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_adv[t1, t2] == g_ret[t2, t1].conj().T).all() for t1, t2 in tt_mesh))

g is Hermitian: False
g_herm is Hermitian: True
True
True
True
True


Extended retarded and advanced parts have step functions omitted from their definitions.

$$
\begin{array}{ll}
    G^{ret,e}(t, t') &= G^>(t, t') - G^<(t, t'),\\
    G^{adv,e}(t, t') &= G^<(t, t') - G^>(t, t').
\end{array}
$$

With help of these auxiliary functions, the Hermitian conjugation is defined in a more straightforward way.
$$
\begin{array}{ll}
    [C^\ddagger]^<_{a,b}(t, t') &= -[C^<_{b,a}(t', t)]^*, \\
    [C^\ddagger]^>_{a,b}(t, t') &= -[C^>_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{ret,e}_{a,b}(t, t') &= -[C^{ret,e}_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{adv,e}_{a,b}(t, t') &= -[C^{adv,e}_{b,a}(t', t)]^*
\end{array}
$$

In [13]:
from tddt.keldysh import retarded_ext, advanced_ext

g_ret_ext, g_adv_ext = retarded_ext(g), advanced_ext(g)
g_hc_ret_ext, g_hc_adv_ext = retarded_ext(g_hc), advanced_ext(g_hc)

print(all((g_hc_ret_ext[t1, t2] == -g_ret_ext[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_adv_ext[t1, t2] == -g_adv_ext[t2, t1].conj().T).all() for t1, t2 in tt_mesh))

True
True


### Arithmetics of `KeldyshGF`

In [14]:
print(g + g)
print(g - g)
print(-g)
print(2 * g)

print(g == g)
print(g == -g)

True
False


### Contraction of discrete indices with an arbitrary tensor

Make a Green's function $G_{ij,kl}(t, t')$ and an array $U_{ijkl}$.

In [15]:
G = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 3), (2, 3)))    
U = np.ones((2, 3, 3, 2))
print(G.arg_index_shapes[0])

(2, 3)


Compute a discrete index contraction $F_{ij,kl}(t, t') = \sum_k U_{ij,nm} G_{mn,kl}(t, t')$.

In [16]:
from tddt.keldysh import target_dot

F = target_dot(G,
               U,
               0,     # Contract over all indices associated with the first time argument of G, i.e. m and n
               (3, 2) # Contract over the 4th and the 3rd indices of U, in this specific order
               )
print(F)

## Convolutions on the contour

<span style="color:red">This functionality is currently broken.</span>

A proper implementation of it could be based on the multi-point contour calculus described in https://iopscience.iop.org/article/10.1088/1751-8121/ab165d.

Contour convolution of two 2-point functions together with discrete index contraction,
$$
    C_{ij,kl}(t, t') = \sum_{mn} \int_\mathcal{C} d\bar t A_{ij,mn}(t, \bar t) B_{mn,kl}(\bar t, t').
$$

In [17]:
# i = 0, 1
# j = 0, 1, 2
# m = 0, 1
# n = 0, 1, 2, 3
# k = 0, 1
# l = 0, 1, 2, 3, 4
A = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 3), (2, 4)))
B = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 4), (2, 5)))

C = A @ B

print(C.n_args)  # Number of time arguments
print(C.arg_index_shapes)

2
((2, 3), (2, 5))


Simultaneous convolution of two 3-point objects over two time arguments and contraction of the respective discrete indices,
$$
    C_{ij,kl}(t, t') = \sum_{mn}\sum_{pq} \int_\mathcal{C} dt_1 dt_2 A_{mn,ij,pq}(t_1, t, t_2) B_{pq,kl,mn}(t_2, t', t_1).
$$

In [18]:
from tddt.keldysh import conv

mesh = MeshProduct(MeshReTime(0.0, t_max, 11),
                   MeshReTime(0.0, t_max, 11),
                   MeshReTime(0.0, t_max, 11))

A = KeldyshGF(mesh=mesh, arg_index_shapes=((2, 3), (2, 4), (2, 5)))
B = KeldyshGF(mesh=mesh, arg_index_shapes=((2, 5), (2, 6), (2, 3)))

C = conv(A, B,
         [(0, 2), (2, 0)])  # Coupling of arguments of A and B: 0 <--> 2, 2 <--> 0 

print(C.n_args)  # Number of time arguments
print(C.arg_index_shapes)

2
((2, 4), (2, 6))


Handling of the non-time mesh components (e.g. momentum/lattice site argument) by `conv()` and `@`:

- If non-time mesh components of `A` and `B` argee, then `C` has the same non-time mesh components.
- Otherwise `C` is defined on a direct product of `A`'s and `B`'s non-time mesh components.

In [19]:
bz_mesh1 = MeshBrillouinZone(bz, 3)
bz_mesh2 = MeshBrillouinZone(bz, 4)

# A and B are defined on the same k-mesh
A = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh1), arg_index_shapes=((2, 2), (2, 2)))
B = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh1), arg_index_shapes=((2, 2), (2, 2)))

C = A @ B
print(len(C.non_time_mesh.components))

# A and B are defined on different k-meshes
A = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh1), arg_index_shapes=((2, 2), (2, 2)))
B = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh2), arg_index_shapes=((2, 2), (2, 2)))

C = A @ B
print(len(C.non_time_mesh.components))

1
2


# 2-nd order dual diagrams for the self-energy and the polarization operator

3-point vertex $\Lambda_{\sigma_1 l_1,\sigma_2 l_2}^{\varsigma l_3 l_4}(t, t', t'')$, fermionic line $G_{\sigma_1 l_1, \sigma_2 l_2}(t, t'; \mathbf{r})$ and bosonic line $W_{\varsigma l_1 l_2, \varsigma' l_3 l_4}(t, t'; \mathbf{r})$.

Here, we use a periodic real-space mesh to turn $\mathbf{k}$- and $\mathbf{q}$-summations in the diagrams into products.

In [20]:
from triqs.gf import MeshCycLat

t_mesh = MeshReTime(0, 5.0, 11)

# Vertex
arg_index_shapes = ((2, 3),    # \sigma_1, l_1
                    (2, 3),    # \sigma_2, l_2
                    (4, 3, 3)) # \varsigma, l_3, l_4

Lambda = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, t_mesh), arg_index_shapes=arg_index_shapes)

n_r = 3  # Number of r-mesh points in each spacial direction
r_mesh = MeshCycLat(lat, n_r)

# Fermionic line
arg_index_shapes = ((2, 3),  # \sigma_1, l_1
                    (2, 3))  # \sigma_2, l_2
G = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, r_mesh),
              arg_index_shapes=arg_index_shapes)

# Bosonic line
arg_index_shapes = ((4, 3, 3),  # \varsigma, l_1, l_2
                    (4, 3, 3))  # \varsigma', l_3, l_4
W = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, r_mesh),
              arg_index_shapes=arg_index_shapes)

# q=0 component of the bosonic line
W_q0 = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh),
                 arg_index_shapes=arg_index_shapes)

In [22]:
from tddt.diagrams import *

Pi = polarization_2nd_order(Lambda, G)              # Eq. (48) in Zhenya's notes
Sigma = selfenergy_2nd_order(Lambda, G, W)          # First line of Eq. (47) in Zhenya's notes
Sigma_HF = selfenergy_2nd_order_hf(Lambda, G, W_q0) # Second line of Eq. (47) in Zhenya's notes

<span style="color:red">N.B. As these functions rely on `tddt.keldysh.conv()`, they currently yield wrong results.</span>

# Contour Dyson-like equation in integral form

`tddt.vie2.solve_vie2()` solves the contour Dyson-like equation in integral form,
$$
    G(t, t') + \int_\mathcal{C} d\bar t F(t, \bar t) G(\bar t, t') = Q(t, t')
$$
under the additional assumptions $Q(t, t') = Q^\ddagger(t, t')$ and

$$
    \int_\mathcal{C} d\bar t F(t, \bar t) Q(\bar t, t') = \int_\mathcal{C} d\bar t Q(t, \bar t) F^\ddagger(\bar t, t').
$$
The contour convolutions imply summations over respective sets of discrete indices. Non-time mesh components of $Q$ and $F$ must agree.

The resulting $G(t, t')$ is also Hermitian, $G(t, t') = G^\ddagger(t, t')$.

In [24]:
t_mesh = MeshReTime(0.0, 5.0, 101)

n_k = 3
bz_mesh = MeshBrillouinZone(bz, n_k)

mesh = MeshProduct(t_mesh, t_mesh, bz_mesh)

# Pauli matrices
s0 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)

# Some simple linear dispersion law \eps(k)
eps_k = np.array([k.value[0] / np.pi + 1.0 for k in bz_mesh])
delta = 1.0

# Fill F(t, t') and Q(t, t')

F_g = Gf(mesh=mesh, target_shape=(2, 2))
F_l = Gf(mesh=mesh, target_shape=(2, 2))
Q_g = Gf(mesh=mesh, target_shape=(2, 2))
Q_l = Gf(mesh=mesh, target_shape=(2, 2))

from scipy.linalg import expm

for k, eps in zip(bz_mesh, eps_k):
    for t1, t2 in MeshProduct(t_mesh, t_mesh):
        dt = t1 - t2
        
        # Define Q via Hamiltonian H(k) = \eps(k) \sigma_0
        H = eps * s0
        e_Q = expm(-1j * H * dt)
        Q_g[t1, t2, k] = -1j * (1 - 0.1) * e_Q
        Q_l[t1, t2, k] = -1j * (-0.1) * e_Q
        
        # Define F via Hamiltonian H = \Delta * (\sigma_x + \sigma_y) / sqrt(2)
        H = delta * (sx + sy) / np.sqrt(2)
        e_F = expm(-1j * H * dt)
        F_g[t1, t2, k] = -1j * (1 - 0.2) * e_F
        F_l[t1, t2, k] = -1j * (-0.2) * e_F

F = from_lesser_greater(F_l, F_g)
Q = from_lesser_greater(Q_l, Q_g)

# Solve the equation

from tddt.vie2 import solve_vie2

G = solve_vie2(F, Q)